# **MESSAGES**

LLMs don't just receive one string — they receive a **list of messages**, each with a role.
LangChain wraps these roles as typed objects so we never mistype a role string.

In [2]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")


Bro API KEY Variable exists


In [3]:
# Import the three message types — each maps to a chat role:
#   SystemMessage  → role: 'system'  (instructions / persona for the AI)
#   HumanMessage   → role: 'user'    (what you / the user says)
#   AIMessage      → role: 'assistant' (what the model previously replied — for multi-turn)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Build a list of messages — this is the 'conversation' sent to the LLM
my_messages = [
    # SystemMessage sets the behaviour / persona.  Sent before user turns.
    SystemMessage(content="You are a gen-z assistant who always answers in a fun way"),
    # HumanMessage is what the user typed
    HumanMessage(content="Bro, tell me a fun fact")
]

# Pass the list to .invoke() — LangChain serialises it to the API format
# .content strips the AIMessage wrapper and gives you just the text
llm_groq.invoke(my_messages).content


'Yaaas, lemme drop some knowledge like a hot new single! Did you know that there\'s a species of jellyfish that\'s IMMORTAL?! Like, for real, the Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. It\'s like a jellyfish version of a reboot, bro! They can just revert back to their polyp stage and grow back into an adult again. Wild, right?'

# **Prompts**

Prompts are *templates* for messages. Instead of hardcoding text, you leave `{placeholders}`
that get filled at runtime — much friendlier than f-strings for LLM inputs.

In [4]:
from langchain_core.prompts import PromptTemplate

user_input = input("Enter a topic for fun fact : ")

# PromptTemplate.from_template() parses {topic} as a variable placeholder
# This is for SINGLE-STRING prompts (no system/user distinction yet)
dynamic_prompt = PromptTemplate.from_template("Write a fun fact about {topic}")

# .invoke({"topic": ...}) substitutes the placeholder → returns a StringPromptValue
# Think of it as "filling in the blank" before sending to the LLM
ready_prompt = dynamic_prompt.invoke({"topic": user_input})

# Now hand the filled prompt to the LLM
llm_groq.invoke(ready_prompt).content


'Fun fact: Aditi Consulting is a global technology consulting firm that was founded in 2000, and its name "Aditi" is derived from Sanskrit, meaning "free" or "unbound", reflecting the company\'s mission to help businesses break free from traditional constraints and achieve innovation and growth through technology.'

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate is the multi-role version of PromptTemplate
# You pass a list of (role, text) tuples — placeholders work in any role
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {tone} assistant"),   # {tone} will be filled at invoke time
    ("user",   "Write a fun fact about {topic}.") # {topic} will also be filled
])

user_input = input("Enter a topic: ")
user_tone  = input("Enter a tone: ")

# .invoke() fills BOTH placeholders at once and returns a ChatPromptValue
# .messages on that object gives you the list of HumanMessage / SystemMessage objects
ready_prompt = prompt_template.invoke({"topic": user_input, "tone": user_tone})

# Pass the list of messages (not the wrapper) to the LLM
llm_groq.invoke(ready_prompt.messages)


AIMessage(content='I\'d love to share a fun fact about Aditi. Aditi is a name that originates from Hindu mythology, and it\'s associated with the goddess of the sky and the earth. In Sanskrit, the word "Aditi" means "free" or "unbound," which is a beautiful and empowering meaning. Isn\'t that a fascinating fact about this lovely name?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 48, 'total_tokens': 124, 'completion_time': 0.266064268, 'completion_tokens_details': None, 'prompt_time': 0.004747103, 'prompt_tokens_details': None, 'queue_time': 0.285062156, 'total_time': 0.270811371}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec27d-5387-7840-9911-32756a545ea8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 76, 'total_tokens': 124})